In [1]:
# Import packages
import os
import datetime
from datetime import datetime
import sys

sys.path.append("/Users/apple/repos/IKIEnv/lib/python3.7/site-packages/")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None

# Impact Analysis (v0.1)

Used to convert results of the 'Trade-Voyage Linking' into impacts on states. Forked from 'evaluation_of_impact_v0.6.ipynb'.

In [2]:
country_iso_codes_cols = ["name", "alpha-2", "alpha-3", "country-code"]
country_iso_codes_dtype = {"alpha-2":str, "alpha-3":str, "country-code":str}
country_iso_codes_renames = {"name":"iso_country", "alpha-2":"iso_2", "alpha-3":"iso_3", "country-code":"iso_code"}

# Read-in ISO Country Labels dataset and use as Standard
country_iso_codes = pd.read_csv("/Users/apple/repos/datasets/national_stats/country_iso_codes.csv", usecols=country_iso_codes_cols, dtype=country_iso_codes_dtype).rename(columns=country_iso_codes_renames)
country_iso_codes.loc[country_iso_codes.iso_country == "Namibia", "iso_2"] = "NA"
country_iso_codes.iloc[:2]

,iso_country,iso_2,iso_3,iso_code
0,Afghanistan,AF,AFG,004
1,Åland Islands,AX,ALA,248


In [3]:
# Finally, include GDP for comparative measures
wb_nat_gdp_cols = ["Country Name", "Country Code", "2018"]
wb_nat_gdp_raw = pd.read_csv("/Users/apple/repos/datasets/national_stats/world_bank_stats/national_gdp/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_5607117.csv", usecols=wb_nat_gdp_cols, skiprows=3)

wb_nat_gdp_raw_2 = pd.merge(wb_nat_gdp_raw, country_iso_codes[["iso_3", "iso_code"]].rename(columns={"iso_3":"Country Code"}), left_on="Country Code", right_on="Country Code", how="left")
wb_nat_gdp = wb_nat_gdp_raw_2[["iso_code", "Country Code", "Country Name", "2018"]].rename(columns={"iso_code":"code_wb", "Country Code":"iso_3_wb", "Country Name":"name_wb", "2018":"gdp_2018_wb"})
wb_nat_gdp.iloc[:2]

,code_wb,iso_3_wb,name_wb,gdp_2018_wb
0,533,ABW,Aruba,3.276188e+09
1,NaN,AFE,Africa Eastern and Southern,1.007196e+12


## Take Aggreated Results Summaries

Notes:
<ul>
    <li>Assumes 50% of MTCs are attributable to Exporters, and 50% are attributable to Importers.</li>
    <li>Assumed the HS2 Commodity Aggregation doesn't need to be halved.</li>
</ul>

### Exporter results

In [4]:
ex_results_d = {"source_iso_code":str, "HS2":str}
ex_results = pd.read_csv("/Users/apple/repos/CaribbEnv/2026/trade_activity_mapping/mapping_ex_trade_ex_voys_info_res_g_coun_hs2.csv", dtype=ex_results_d)
display(ex_results.head(2))

# Derive results for Exporters
ex_results_agg_a = {
    "vol_kg":"sum", "val_usd":"sum", 
    "vol_kg_recon":"sum", "val_usd_recon":"sum", "ene_tj_recon":"sum", "co2_t_recon":"sum", 
    "ets_23_usd_recon":"sum", "ets_30_usd_recon":"sum",
    "bau_usd_23_recon":"sum", "bau_usd_30_recon":"sum", "bau_usd_40_recon":"sum", "bau_usd_50_recon":"sum",
    "s24_usd_23_recon":"sum", "s24_usd_30_recon":"sum", "s24_usd_40_recon":"sum", "s24_usd_50_recon":"sum"
}
ex_results_agg = ex_results.groupby(["source_iso_code", "source_country"]).agg(ex_results_agg_a).reset_index()
ex_results_agg_wb = pd.merge(
    wb_nat_gdp, ex_results_agg, 
    left_on="code_wb", right_on="source_iso_code", 
    how="right").drop(columns=["code_wb", "iso_3_wb", "name_wb"])

## Summary Statistics
# Evaluate Reconstruction Performance by Country
ex_results_agg_wb["pct_vol_recon"] = round(100 * ex_results_agg_wb.vol_kg_recon / ex_results_agg_wb.vol_kg, 2)
ex_results_agg_wb["pct_val_recon"] = round(100 * ex_results_agg_wb.val_usd_recon / ex_results_agg_wb.val_usd, 2)

# ETS Costs in 2030
ex_results_agg_wb["ets_23_pct_gdp"] = round(100 * ex_results_agg_wb.ets_23_usd_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["ets_30_pct_gdp"] = round(100 * ex_results_agg_wb.ets_30_usd_recon / ex_results_agg_wb.gdp_2018_wb, 5)

# NZF Costs - Pre-evaluate NZF Deltas between S24 and BAU 
ex_results_agg_wb["s24_usd_23_recon_del"] = ex_results_agg_wb.s24_usd_23_recon - ex_results_agg_wb.bau_usd_23_recon
ex_results_agg_wb["s24_usd_30_recon_del"] = ex_results_agg_wb.s24_usd_30_recon - ex_results_agg_wb.bau_usd_30_recon
ex_results_agg_wb["s24_usd_40_recon_del"] = ex_results_agg_wb.s24_usd_40_recon - ex_results_agg_wb.bau_usd_40_recon
ex_results_agg_wb["s24_usd_50_recon_del"] = ex_results_agg_wb.s24_usd_50_recon - ex_results_agg_wb.bau_usd_50_recon

# BAU Costs
ex_results_agg_wb["bau_23_pct_gdp"] = round(100 * ex_results_agg_wb.bau_usd_23_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["bau_30_pct_gdp"] = round(100 * ex_results_agg_wb.bau_usd_30_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["bau_40_pct_gdp"] = round(100 * ex_results_agg_wb.bau_usd_40_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["bau_50_pct_gdp"] = round(100 * ex_results_agg_wb.bau_usd_50_recon / ex_results_agg_wb.gdp_2018_wb, 5)

# S24 Costs
ex_results_agg_wb["s24_23_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_23_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["s24_30_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_30_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["s24_40_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_40_recon / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["s24_50_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_50_recon / ex_results_agg_wb.gdp_2018_wb, 5)

# S24 Delta Costs
ex_results_agg_wb["s24_del_23_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_23_recon_del / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["s24_del_30_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_30_recon_del / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["s24_del_40_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_40_recon_del / ex_results_agg_wb.gdp_2018_wb, 5)
ex_results_agg_wb["s24_del_50_pct_gdp"] = round(100 * ex_results_agg_wb.s24_usd_50_recon_del / ex_results_agg_wb.gdp_2018_wb, 5)

ex_results_agg_wb.to_csv("/Users/apple/repos/CaribbEnv/2026/impact_analysis/impact_analysis_ex.csv", index=False)
ex_results_agg_wb.head(2)

,source_iso_code,source_country,HS2,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,004,Afghanistan,02,19970.561,3789.508,3789.508,19970.561,0.000597,0.045780,0.0,0.0,11.313996,10.345822,9.952322,9.471194,11.313996,12.436627,17.303836,18.202640
1,004,Afghanistan,04,2592.219,969.702,969.702,2592.219,0.000048,0.003657,0.0,0.0,1.344396,1.219692,1.155249,1.090276,1.344396,1.463483,2.045604,2.153784


,gdp_2018_wb,source_iso_code,source_country,vol_kg,val_usd,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon,pct_vol_recon,pct_val_recon,ets_23_pct_gdp,ets_30_pct_gdp,s24_usd_23_recon_del,s24_usd_30_recon_del,s24_usd_40_recon_del,s24_usd_50_recon_del,bau_23_pct_gdp,bau_30_pct_gdp,bau_40_pct_gdp,bau_50_pct_gdp,s24_23_pct_gdp,s24_30_pct_gdp,s24_40_pct_gdp,s24_50_pct_gdp,s24_del_23_pct_gdp,s24_del_30_pct_gdp,s24_del_40_pct_gdp,s24_del_50_pct_gdp
0,1.841886e+10,004,Afghanistan,3.215669e+08,1.697071e+08,3.215669e+08,1.697071e+08,12.663680,977.624399,0.0,5.961390e+02,4.793903e+05,4.618930e+05,4.050516e+05,3.874610e+05,4.793903e+05,5.354202e+05,6.758048e+05,7.112176e+05,100.0,100.0,0.0,0.00000,0.0,7.352728e+04,2.707532e+05,3.237566e+05,0.00260,0.00251,0.00220,0.00210,0.00260,0.00291,0.00367,0.00386,0.0,0.00040,0.00147,0.00176
1,1.515642e+10,008,Albania,1.820407e+09,4.759043e+08,1.820407e+09,4.759043e+08,213.669384,16440.066449,0.0,3.519236e+06,1.102927e+07,1.071760e+07,9.747937e+06,9.388475e+06,1.102927e+07,1.185084e+07,1.431403e+07,1.499985e+07,100.0,100.0,0.0,0.02322,0.0,1.133242e+06,4.566092e+06,5.611375e+06,0.07277,0.07071,0.06432,0.06194,0.07277,0.07819,0.09444,0.09897,0.0,0.00748,0.03013,0.03702


### Importer results

In [5]:
im_results_d = {"dest_iso_code":str, "HS2":str}
im_results = pd.read_csv("/Users/apple/repos/CaribbEnv/2026/trade_activity_mapping/mapping_im_trade_im_voys_info_res_g_coun_hs2.csv", dtype=im_results_d)
display(im_results.head(2))

# Derive results for Exporters
im_results_agg_a = {
    "vol_kg":"sum", "val_usd":"sum", 
    "vol_kg_recon":"sum", "val_usd_recon":"sum", "ene_tj_recon":"sum", "co2_t_recon":"sum", 
    "ets_23_usd_recon":"sum", "ets_30_usd_recon":"sum",
    "bau_usd_23_recon":"sum", "bau_usd_30_recon":"sum", "bau_usd_40_recon":"sum", "bau_usd_50_recon":"sum",
    "s24_usd_23_recon":"sum", "s24_usd_30_recon":"sum", "s24_usd_40_recon":"sum", "s24_usd_50_recon":"sum"
}
im_results_agg = im_results.groupby(["dest_iso_code", "dest_country"]).agg(im_results_agg_a).reset_index()
im_results_agg_wb = pd.merge(
    wb_nat_gdp, im_results_agg, 
    left_on="code_wb", right_on="dest_iso_code", 
    how="right").drop(columns=["code_wb", "iso_3_wb", "name_wb"])

## Summary Statistics
# Evaluate Reconstruction Performance by Country
im_results_agg_wb["pct_vol_recon"] = round(100 * im_results_agg_wb.vol_kg_recon / im_results_agg_wb.vol_kg, 2)
im_results_agg_wb["pct_val_recon"] = round(100 * im_results_agg_wb.val_usd_recon / im_results_agg_wb.val_usd, 2)

# ETS Costs in 2030
im_results_agg_wb["ets_23_pct_gdp"] = round(100 * im_results_agg_wb.ets_23_usd_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["ets_30_pct_gdp"] = round(100 * im_results_agg_wb.ets_30_usd_recon / im_results_agg_wb.gdp_2018_wb, 5)

# NZF Costs - Pre-evaluate NZF Deltas between S24 and BAU 
im_results_agg_wb["s24_usd_23_recon_del"] = im_results_agg_wb.s24_usd_23_recon - im_results_agg_wb.bau_usd_23_recon
im_results_agg_wb["s24_usd_30_recon_del"] = im_results_agg_wb.s24_usd_30_recon - im_results_agg_wb.bau_usd_30_recon
im_results_agg_wb["s24_usd_40_recon_del"] = im_results_agg_wb.s24_usd_40_recon - im_results_agg_wb.bau_usd_40_recon
im_results_agg_wb["s24_usd_50_recon_del"] = im_results_agg_wb.s24_usd_50_recon - im_results_agg_wb.bau_usd_50_recon

# BAU Costs
im_results_agg_wb["bau_23_pct_gdp"] = round(100 * im_results_agg_wb.bau_usd_23_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["bau_30_pct_gdp"] = round(100 * im_results_agg_wb.bau_usd_30_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["bau_40_pct_gdp"] = round(100 * im_results_agg_wb.bau_usd_40_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["bau_50_pct_gdp"] = round(100 * im_results_agg_wb.bau_usd_50_recon / im_results_agg_wb.gdp_2018_wb, 5)

# S24 Costs
im_results_agg_wb["s24_23_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_23_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["s24_30_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_30_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["s24_40_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_40_recon / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["s24_50_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_50_recon / im_results_agg_wb.gdp_2018_wb, 5)

# S24 Delta Costs
im_results_agg_wb["s24_del_23_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_23_recon_del / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["s24_del_30_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_30_recon_del / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["s24_del_40_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_40_recon_del / im_results_agg_wb.gdp_2018_wb, 5)
im_results_agg_wb["s24_del_50_pct_gdp"] = round(100 * im_results_agg_wb.s24_usd_50_recon_del / im_results_agg_wb.gdp_2018_wb, 5)

im_results_agg_wb.to_csv("/Users/apple/repos/CaribbEnv/2026/impact_analysis/impact_analysis_im.csv", index=False)
im_results_agg_wb.head(2)

,dest_iso_code,dest_country,HS2,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,004,Afghanistan,02,654723.341,743604.476,743604.476,654723.341,0.670863,51.911522,0.0,0.0,18547.374117,15389.628966,14107.304632,13588.441648,18547.374117,20362.912815,27380.436828,28592.600593
1,004,Afghanistan,03,2578345.296,2058436.090,2058436.090,2578345.296,0.097763,7.549184,0.0,0.0,3192.415452,2805.759370,2615.775169,2487.235323,3192.415452,3490.490995,4820.101740,5074.506603


,gdp_2018_wb,dest_iso_code,dest_country,vol_kg,val_usd,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon,pct_vol_recon,pct_val_recon,ets_23_pct_gdp,ets_30_pct_gdp,s24_usd_23_recon_del,s24_usd_30_recon_del,s24_usd_40_recon_del,s24_usd_50_recon_del,bau_23_pct_gdp,bau_30_pct_gdp,bau_40_pct_gdp,bau_50_pct_gdp,s24_23_pct_gdp,s24_30_pct_gdp,s24_40_pct_gdp,s24_50_pct_gdp,s24_del_23_pct_gdp,s24_del_30_pct_gdp,s24_del_40_pct_gdp,s24_del_50_pct_gdp
0,1.841886e+10,004,Afghanistan,3.425029e+09,9.186016e+08,3.425029e+09,9.186016e+08,308.781841,23800.721797,0.0,4.327305e+04,1.229163e+07,1.192294e+07,1.061062e+07,1.034061e+07,1.229163e+07,1.359457e+07,1.667624e+07,1.758158e+07,100.0,100.0,0.0,0.00023,0.0,1.671623e+06,6.065614e+06,7.240971e+06,0.06673,0.06473,0.05761,0.05614,0.06673,0.07381,0.09054,0.09545,0.0,0.00908,0.03293,0.03931
1,1.515642e+10,008,Albania,6.052463e+08,1.115782e+09,6.052463e+08,1.115782e+09,111.282450,8503.000761,0.0,1.587598e+06,5.311860e+06,5.102254e+06,4.810046e+06,4.695086e+06,5.311860e+06,5.861491e+06,7.102178e+06,7.437007e+06,100.0,100.0,0.0,0.01047,0.0,7.592374e+05,2.292132e+06,2.741920e+06,0.03505,0.03366,0.03174,0.03098,0.03505,0.03867,0.04686,0.04907,0.0,0.00501,0.01512,0.01809
